# Polars Exercises 10 — Capstone: GDP Case Study

The final mission. We bring together **everything** — reading messy files, cleaning, casting, renaming, window functions, pivoting, and multi-key joins — to build a real analytics report from three World-Bank / OECD datasets: **GDP**, **population**, and **R&D expenditure**. The goal: GDP per capita and R&D spend as a share of GDP, per country and year. This is a complete Foundry transform, start to finish.

*המשימה האחרונה. נחבר את הכל — קריאת קבצים מלוכלכים, ניקוי, המרת טיפוסים, שינוי שמות, window functions, pivot, ו-joins על כמה מפתחות — כדי לבנות דוח אנליטי אמיתי משלושה מקורות: GDP, אוכלוסייה, והוצאות מו"פ. זו טרנספורמציה שלמה מתחילתה ועד סופה.*

**Data files:** `data/gdp.csv`, `data/population.csv`, `data/expenditure.csv`

---

> **גרסת פתרונות** — נסו קודם לבד מתוך `Exercises.ipynb`.

### 1. Import Polars. Read `data/gdp.csv` and **rename** the columns to `country`, `code`, `year`, `gdp`. Save as `gdp` and show the head.

*ייבאו את Polars. קראו את `data/gdp.csv` ושנו את שמות העמודות ל-`country`, `code`, `year`, `gdp`. שמרו כ-`gdp`.*

In [1]:
import polars as pl

gdp = pl.read_csv("data/gdp.csv").rename({
    "Country Name": "country", "Country Code": "code",
    "Year": "year", "Value": "gdp",
})
gdp.head()

country,code,year,gdp
str,str,i64,f64
"""Afghanistan""","""AFG""",2000,3.5214e9
"""Afghanistan""","""AFG""",2001,2.8136e9
"""Afghanistan""","""AFG""",2002,3.8257e9
"""Afghanistan""","""AFG""",2003,4.5209e9
"""Afghanistan""","""AFG""",2004,5.2249e9


### 2. Explore: run `describe()` on `gdp`. Over how many years does the data span (min and max `year`)?

*חקרו: הריצו `describe()` על `gdp`. על פני כמה שנים פרוש המידע (min ו-max של `year`)?*

In [2]:
gdp.describe()

statistic,country,code,year,gdp
str,str,str,f64,f64
"""count""","""13979""","""13979""",13979.0,13979.0
"""null_count""","""0""","""0""",0.0,0.0
"""mean""",null,null,1994.672866,1.2074e12
"""std""",null,null,17.731413,5.5375e12
"""min""","""Afghanistan""","""ABW""",1960.0,11502.632645
"""25%""",null,null,1980.0,2.2348e9
"""50%""",null,null,1996.0,1.6726e10
"""75%""",null,null,2010.0,2.0613e11
"""max""","""Zimbabwe""","""ZWE""",2023.0,1.0544e14


### 3. What is the **single largest GDP value** in the whole dataset, and which `country` and `year` does it belong to?

*מהו ערך ה-GDP הגדול ביותר בכל המידע, ולאיזו `country` ו-`year` הוא שייך?*

In [3]:
top = gdp.select(pl.col("gdp").max()).item()
gdp.filter(pl.col("gdp") == top)

country,code,year,gdp
str,str,i64,f64
"""World""","""WLD""",2023,1.0544e14


### 4. How many years of data exist **per country**? Then: how many countries have **fewer than 64** years of data?

*כמה שנות מידע יש לכל מדינה? וכמה מדינות יש להן פחות מ-64 שנות מידע?*

In [4]:
counts = gdp.group_by("country", "code").agg(pl.len())
counts.filter(pl.col("len") < 64).height

128

### 5. **Window functions.** For each country (`over("code")`), add four columns: `worst_gdp`, `worst_year`, `best_gdp`, `best_year` — the lowest and highest GDP and the years they happened. Save back to `gdp`.

*window functions. לכל מדינה (`over("code")`), הוסיפו ארבע עמודות: `worst_gdp`, `worst_year`, `best_gdp`, `best_year`. שמרו ל-`gdp`.*

In [5]:
gdp = gdp.with_columns(
    worst_gdp=pl.col("gdp").get(pl.col("gdp").arg_min()).over("code"),
    worst_year=pl.col("year").get(pl.col("gdp").arg_min()).over("code"),
    best_gdp=pl.col("gdp").get(pl.col("gdp").arg_max()).over("code"),
    best_year=pl.col("year").get(pl.col("gdp").arg_max()).over("code"),
)
gdp.filter(pl.col("code") == "AFG").head()

country,code,year,gdp,worst_gdp,worst_year,best_gdp,best_year
str,str,i64,f64,f64,i64,f64,i64
"""Afghanistan""","""AFG""",2000,3.5214e9,2.8136e9,2001,2.0497e10,2014
"""Afghanistan""","""AFG""",2001,2.8136e9,2.8136e9,2001,2.0497e10,2014
"""Afghanistan""","""AFG""",2002,3.8257e9,2.8136e9,2001,2.0497e10,2014
"""Afghanistan""","""AFG""",2003,4.5209e9,2.8136e9,2001,2.0497e10,2014
"""Afghanistan""","""AFG""",2004,5.2249e9,2.8136e9,2001,2.0497e10,2014


### 6. **Pivot** the original GDP figures into a wide table: one row per `country` + `code`, one column per `year`, values are `gdp`. Save as `gdp_pivot` and show its shape.

*pivot של נתוני ה-GDP לטבלה רחבה: שורה לכל `country`+`code`, עמודה לכל `year`, הערכים הם `gdp`. שמרו כ-`gdp_pivot`.*

In [6]:
gdp_pivot = gdp.pivot(
    on="year", index=["country", "code"], values="gdp", sort_columns=True
)
gdp_pivot.shape

(262, 66)

### 7. Add a `best_gdp` column to `gdp_pivot` = the maximum across all the year columns (use `pl.max_horizontal` over the year columns).

*הוסיפו ל-`gdp_pivot` עמודת `best_gdp` = המקסימום על פני כל עמודות השנים (השתמשו ב-`pl.max_horizontal`).*

In [7]:
year_cols = [c for c in gdp_pivot.columns if c not in ("country", "code")]
gdp_pivot = gdp_pivot.with_columns(best_gdp=pl.max_horizontal(year_cols))
gdp_pivot.select("country", "code", "best_gdp").head()

country,code,best_gdp
str,str,f64
"""Afghanistan""","""AFG""",2.0497e10
"""Africa Eastern and Southern""","""AFE""",1.2362e12
"""Africa Western and Central""","""AFW""",8.9459e11
"""Albania""","""ALB""",2.2978e10
"""Algeria""","""DZA""",2.3990e11


### 8. Write `gdp_pivot` to `processed/gdp_pivot.csv` (create the `processed` folder first).

*כתבו את `gdp_pivot` ל-`processed/gdp_pivot.csv` (צרו קודם את התיקייה).*

In [8]:
from pathlib import Path
Path("processed").mkdir(exist_ok=True)
gdp_pivot.write_csv("processed/gdp_pivot.csv")
print("wrote processed/gdp_pivot.csv")

wrote processed/gdp_pivot.csv


### 9. Read `data/population.csv`. Its `Value` column looks like integers at first but has decimals later, so the default type inference fails. Read it robustly with `infer_schema_length=10000` and check the schema.

*קראו את `data/population.csv`. עמודת `Value` נראית כמו מספרים שלמים בהתחלה אבל יש בה עשרוניים בהמשך, ולכן זיהוי הטיפוס האוטומטי נכשל. קראו אותה בעזרת `infer_schema_length=10000`.*

In [9]:
pop = pl.read_csv("data/population.csv", infer_schema_length=10000)
pop.schema

Schema([('Country Name', String),
        ('Country Code', String),
        ('Year', Int64),
        ('Value', Float64)])

### 10. Clean `pop`: rename `Country Name`→`country`, `Country Code`→`code`, `Year`→`year`, cast the population `Value` to an integer column `pop`, and keep only `country, code, year, pop`.

*נקו את `pop`: שנו שמות, המירו את `Value` לעמודה שלמה `pop`, והשאירו רק `country, code, year, pop`.*

In [10]:
pop = pop.rename({
    "Country Name": "country", "Country Code": "code", "Year": "year"
}).with_columns(
    pop=pl.col("Value").cast(pl.Int64)
).select("country", "code", "year", "pop")
pop.head()

country,code,year,pop
str,str,i64,i64
"""Aruba""","""ABW""",1960,54922
"""Aruba""","""ABW""",1961,55578
"""Aruba""","""ABW""",1962,56320
"""Aruba""","""ABW""",1963,57002
"""Aruba""","""ABW""",1964,57619


### 11. **Join** the population into `gdp` on the three keys `["country", "code", "year"]` (left join). Save back to `gdp`.

*חברו (join) את האוכלוסייה לתוך `gdp` על שלושת המפתחות (left join). שמרו ל-`gdp`.*

In [11]:
gdp = gdp.join(pop, on=["country", "code", "year"], how="left")
gdp.select("country", "year", "gdp", "pop").head()

country,year,gdp,pop
str,i64,f64,i64
"""Afghanistan""",2000,3.5214e9,20130327
"""Afghanistan""",2001,2.8136e9,20284307
"""Afghanistan""",2002,3.8257e9,21378117
"""Afghanistan""",2003,4.5209e9,22733049
"""Afghanistan""",2004,5.2249e9,23560654


### 12. Compute **GDP per capita**: add a column `capita` = `gdp / pop`. Save back to `gdp`.

*חשבו תוצר לנפש: הוסיפו עמודה `capita` = `gdp / pop`. שמרו ל-`gdp`.*

In [12]:
gdp = gdp.with_columns(capita=pl.col("gdp") / pl.col("pop"))
gdp.select("country", "year", "gdp", "pop", "capita").head()

country,year,gdp,pop,capita
str,i64,f64,i64,f64
"""Afghanistan""",2000,3.5214e9,20130327,174.930991
"""Afghanistan""",2001,2.8136e9,20284307,138.706822
"""Afghanistan""",2002,3.8257e9,21378117,178.954088
"""Afghanistan""",2003,4.5209e9,22733049,198.871116
"""Afghanistan""",2004,5.2249e9,23560654,221.763654


### 13. Quality check: are there suspicious per-capita values? Show rows where `capita` is above 200,000 (likely data quirks or tiny states).

*בדיקת איכות: יש ערכי תוצר-לנפש חשודים? הציגו שורות שבהן `capita` מעל 200,000.*

In [13]:
gdp.filter(pl.col("capita") > 200_000).select("country", "year", "capita").head()

country,year,capita
str,i64,f64
"""Monaco""",2008,204263.797114
"""Monaco""",2021,223897.041568
"""Monaco""",2022,225630.036004


### 14. Write the current `gdp` table (with population & per-capita) to `processed/gdp_report.csv`.

*כתבו את טבלת `gdp` הנוכחית ל-`processed/gdp_report.csv`.*

In [14]:
gdp.write_csv("processed/gdp_report.csv")
print("wrote processed/gdp_report.csv")

wrote processed/gdp_report.csv


### 15. Read `data/expenditure.csv`, keeping only the `LOCATION`, `TIME` and `Government` columns, and rename them to `code`, `year`, `spend`. Save as `rd`.

*קראו את `data/expenditure.csv`, השאירו רק `LOCATION`, `TIME`, `Government`, ושנו שמות ל-`code`, `year`, `spend`. שמרו כ-`rd`.*

In [15]:
rd = pl.read_csv("data/expenditure.csv").select(
    code=pl.col("LOCATION"),
    year=pl.col("TIME"),
    spend=pl.col("Government"),
)
rd.head()

code,year,spend
str,i64,f64
"""ALB""",2007,15068.61246
"""ALB""",2008,30201.12059
"""ARE""",2014,1.0928e6
"""ARG""",1996,null
"""ARG""",1997,1.1531e6


### 16. The R&D figures are reported in thousands. Convert to absolute spend: add `k_spend` = `spend * 1000`, then keep only `code, year, k_spend`.

*נתוני המו"פ מדווחים באלפים. המירו לסכום מלא: הוסיפו `k_spend` = `spend * 1000`, והשאירו רק `code, year, k_spend`.*

In [16]:
rd = rd.with_columns(k_spend=pl.col("spend") * 1000)\
       .select("code", "year", "k_spend")
rd.head()

code,year,k_spend
str,i64,f64
"""ALB""",2007,1.5069e7
"""ALB""",2008,3.0201e7
"""ARE""",2014,1.0928e9
"""ARG""",1996,null
"""ARG""",1997,1.1531e9


### 17. **Join** `rd` into `gdp` on `["code", "year"]` (left join). Save back to `gdp`.

*חברו את `rd` לתוך `gdp` על `["code", "year"]` (left join). שמרו ל-`gdp`.*

In [17]:
gdp = gdp.join(rd, on=["code", "year"], how="left")
gdp.filter(pl.col("code") == "USA").select("year", "gdp", "k_spend").tail(5)

year,gdp,k_spend
i64,f64,f64
2019,2.1521e13,null
2020,2.1323e13,null
2021,2.3594e13,null
2022,2.5744e13,null
2023,2.7361e13,null


### 18. Compute R&D as a **share of GDP**: add `spend_pct` = `k_spend / gdp`. Save back to `gdp`.

*חשבו את המו"פ כאחוז מהתוצר: הוסיפו `spend_pct` = `k_spend / gdp`. שמרו ל-`gdp`.*

In [18]:
gdp = gdp.with_columns(spend_pct=pl.col("k_spend") / pl.col("gdp"))
gdp.filter(pl.col("code") == "USA").select("year", "gdp", "k_spend", "spend_pct").tail(5)

year,gdp,k_spend,spend_pct
i64,f64,f64,f64
2019,2.1521e13,null,null
2020,2.1323e13,null,null
2021,2.3594e13,null,null
2022,2.5744e13,null,null
2023,2.7361e13,null,null


### 19. Write the full enriched report to `processed/gdp_full_report.csv`.

*כתבו את הדוח המלא והמועשר ל-`processed/gdp_full_report.csv`.*

In [19]:
gdp.write_csv("processed/gdp_full_report.csv")
print("wrote processed/gdp_full_report.csv")

wrote processed/gdp_full_report.csv


### 20. 🎯 **Insight 1:** the **top 10 countries by GDP per capita in 2014** (filter year, sort, head).

*תובנה 1: 10 המדינות המובילות לפי תוצר-לנפש בשנת 2014.*

In [20]:
gdp.filter((pl.col("year") == 2014) & pl.col("capita").is_not_null())\
   .sort("capita", descending=True)\
   .select("country", "capita").head(10)

country,capita
str,f64
"""Monaco""",195675.184707
"""Liechtenstein""",178735.153021
"""Luxembourg""",123678.702143
"""Bermuda""",100961.576603
"""Norway""",97666.695184
"""Qatar""",95840.631009
"""Isle of Man""",91885.452872
"""Switzerland""",88724.99094
"""Macao SAR, China""",88310.811957


### 21. 🎯 **Insight 2:** among rows that *have* R&D data, the **top 10 by `spend_pct`** (R&D as a share of GDP) — who invests most in research?

*תובנה 2: מבין השורות שיש בהן נתוני מו"פ, 10 המובילות לפי `spend_pct` — מי משקיע הכי הרבה במחקר?*

In [21]:
gdp.filter(pl.col("spend_pct").is_not_null())\
   .sort("spend_pct", descending=True)\
   .select("country", "year", "spend_pct").head(10)

country,year,spend_pct
str,i64,f64
"""Russian Federation""",2002,0.024623
"""Russian Federation""",2003,0.023843
"""Russian Federation""",2001,0.023622
"""Belarus""",2001,0.023343
"""Iran, Islamic Rep.""",2003,0.023235
"""Russian Federation""",1999,0.022641
"""Russian Federation""",2000,0.022168
"""Ukraine""",2001,0.021825
"""Russian Federation""",2015,0.02066


### 22. 🎯 **Insight 3:** for the USA, the **average `spend_pct` across all years that have data** (filter USA + non-null, then mean).

*תובנה 3: עבור ארה"ב, ה-`spend_pct` הממוצע על פני כל השנים עם נתונים.*

In [22]:
gdp.filter((pl.col("code") == "USA") & pl.col("spend_pct").is_not_null())\
   .select(pl.col("spend_pct").mean().alias("usa_avg_spend_pct"))

usa_avg_spend_pct
f64
0.007795
